# Core Machine Learning — Part 2: Supervised & Unsupervised Learning (with STAR)

*An improved, interactive follow-up to **core_ml_part1.ipynb**.*

Part 1 gave you the big map (data prep, model selection, evaluation, bias–variance,
interpretability, deployment). This notebook **zooms into the two branches of ML that
interviewers ask about most** — **supervised learning** (we have labelled answers to learn
from) and **unsupervised learning** (we search for hidden structure with no labels).

Every key concept is explained with a **STAR** structure so you can tell *a complete story*
in an interview instead of reciting a definition:

| Letter | Meaning | Question it answers |
|---|---|---|
| **S** — Situation | What is the problem? | Why do we need ML at all here? |
| **T** — Task | What must we achieve? | What exactly are we predicting/discovering? |
| **A** — Action | How can this be resolved? | Which algorithm + approach do we use? |
| **R** — Result | Success criteria | How do we prove the approach worked? |

Everything here runs on **simulated data** (generated on the fly with `sklearn`), uses
real graphs, and needs **no external files**. Just run every cell top-to-bottom.

## Table of Contents
1. **Supervised vs Unsupervised** — the big picture and how to choose
2. **Supervised · Regression** — Linear Regression (predict a number)
3. **Supervised · Classification** — Logistic Regression + ROC/AUC (predict a category)
4. **Supervised · k-Nearest Neighbours** — classify by similarity
5. **Supervised · Decision Trees & Random Forest** — interpretable rules + robust ensemble
6. **Validation** — train/test split & cross-validation (never trust one score)
7. **Unsupervised · Clustering** — k-Means, DBSCAN, Hierarchical (find groups)
8. **Unsupervised · Dimensionality Reduction** — PCA (compress features safely)
9. **Cheat-sheet** — problem → algorithm → metric (the interview recap)


## Before we start — the STAR "recipe" you will reuse everywhere

Interviewers love structure. For **any** ML question, answer in four moves:

1. **Situation — name the real-world problem.**
   *Example:* "A bank receives thousands of loan applications but default rates keep
   rising because decisions are made by hand and inconsistently."
2. **Task — translate the problem into an ML task.**
   *Example:* "Predict, for each applicant, the probability that they will default within
   12 months, given their income, credit history, age, and debt ratio."
3. **Action — choose the approach and justify it.**
   *Example:* "Logistic Regression as a fast, interpretable baseline, then Random Forest
   if more accuracy is needed; evaluate with cross-validation on historical labelled data."
4. **Result — state the success criteria.**
   *Example:* "Success = ROC-AUC > 0.8 on held-out data *and* higher real default
   recovery than the current manual process, *while* remaining explainable to regulators."

Each section below applies exactly this template. When you can answer "how do I prove it
worked?" for every algorithm, you are ready for the interview.


## 1. Supervised vs Unsupervised — the big picture

**Why it matters:** *before* choosing an algorithm, you must decide **which family** of
algorithms your problem belongs to. Hiring managers want to hear you reason about labels,
not just name models.

### 🧠 Beginner's guide
A **supervised** problem is like a classroom where the student is handed **flashcards with
the answers on the back** (`labels` / `targets`). The model learns a mapping
`features → answer`, then predicts answers for new cards.

An **unsupervised** problem is like giving a cartographer a pile of **unlabelled dots** and
asking them to *find islands and coastlines*. There are no answers — only structure.

| | **Supervised** | **Unsupervised** |
|---|---|---|
| Data has labels (`y`)? | ✅ Yes | ❌ No |
| What it does | Learns `features → label` | Finds hidden structure in features |
| Two main flavours | **Regression** (number) + **Classification** (category) | **Clustering** (groups) + **Dimensionality reduction** (compress) |
| Examples | House price, email spam, loan default | Customer segments, anomaly detection, compressing 1000 features |
| Typical output metric | Accuracy, F1, R², ROC-AUC | Silhouette score, explained variance, cluster stability |

### ⭐ STAR — Choosing the ML **family** (supervised vs unsupervised)
| Field | Content |
|---|---|
| **S**ituation | We have a dataset but don't know whether we can train a model. Do points carry an answer (label) or not? |
| **T**ask | Decide whether we are *predicting* (supervised) or *discovering* (unsupervised), then map the data accordingly. |
| **A**ction | If a reliable target `y` exists → supervised (regression for continuous `y`, classification for categorical `y`). If only features exist, or labels are too costly to collect → unsupervised (clustering / PCA). |
| **R**esult (**success criteria**) | **Supervised:** a held-out score beats a simple baseline (e.g. accuracy > random, R² meaningfully > 0). **Unsupervised:** clusters are compact & separated (high silhouette), or few PCA components still preserve ~95% of variance. |


In [ ]:
# --- Setup: imports, reproducibility, and simulated data ---
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_regression, make_classification, make_blobs, make_moons
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    mean_absolute_error, mean_squared_error, root_mean_squared_error,
    r2_score, roc_curve, roc_auc_score, silhouette_score,
)

sns.set_theme(style="whitegrid")
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)

print("Setup ready. Every dataset in this notebook is SIMULATED on the fly —")
print("you can run this notebook anywhere without downloading any data.")


In [ ]:
# --- 1. Supervised vs Unsupervised at a glance ---
# A SUPERVISED problem has a label/target (y). An UNSUPERVISED problem has only features (X).

X_sup, y_sup = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                   n_informative=2, n_clusters_per_class=1,
                                   class_sep=1.1, random_state=42)

# Unsupervised: the same data, but the label is "hidden" (we pretend we never saw y)
X_unsup = X_sup.copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_sup[:, 0], X_sup[:, 1], c=y_sup, cmap="coolwarm", edgecolor="k", s=40)
axes[0].set_title("Supervised: labels KNOWN (blue vs red)")
axes[1].scatter(X_unsup[:, 0], X_unsup[:, 1], c="slategray", edgecolor="k", s=40)
axes[1].set_title("Unsupervised: only features, NO labels")
for ax in axes:
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
plt.tight_layout()
plt.show()


### Discussion — Supervised vs Unsupervised (interview favourites)
- **"How do you decide between supervised and unsupervised learning?"** → First ask: *do I
  have labels?* If yes and reliable → supervised. If labels are expensive, missing, or the
  goal is exploration → unsupervised. *Beginner tip: don't overthink. A typical sales pitch:
  "If I have historical 'did the customer buy?' data, I supervise a classifier; if I only
  have customer profiles, I cluster them into segments."*
- **"Can you ever use them together?"** → Yes. It's called **semi-supervised / self-supervised**.
  Often you cluster first (unsupervised) to discover segments, *then* hand-label a few
  examples per cluster and train a supervised classifier with very few labels. GANs and
  autoencoders are unsupervised; the LLMs in modern AI are mostly self-supervised (predict
  the next word — labels come from the text itself).


## 2. Supervised · Regression — Linear Regression

**Why it matters:** Regression is the simplest supervised task and the very first thing
interviewers test. If you can explain *why* the best-fit line is "best", you look sharp.

### 🧠 Beginner's guide
Imagine predicting a house's price from its size. You place every (`size`, `price`) pair on
a scatter plot and draw the straight line that comes `closest` to all points. That line is
`price = b0 + b1 * size` where:
- `b1` (the **slope / coefficient**) = how much price changes for each extra square metre,
- `b0` (the **intercept**) = the price when size is zero (usually just a baseline to anchor
  the line).

"Closest" is measured by **least squares**: we minimise the sum of squared vertical
distances between the points and the line. Squaring makes big mistakes hurt more and keeps
errors positive.

### ⭐ STAR — Linear Regression
| Field | Content |
|---|---|
| **S**ituation | We must predict a **continuous number** (house price, temperature, revenue next quarter). The relationship between features and target looks roughly straight-line. |
| **T**ask | Learn the line `y = b0 + b1·x` that minimises prediction error on new, unseen points. |
| **A**ction | Use **Ordinary Least Squares (OLS)** to find `b0`, `b1` that minimise `Σ(y − ŷ)²`. Split data into train/test so we measure *generalisation*, not memorisation. Diagnose with residuals (no patterns → linearity holds). |
| **R**esult (**success criteria**) | High **R²** (close to 1 = the model explains most variance; meaningfully better than 0), **low MAE/RMSE** in original units, residuals randomly scattered around zero, and interpretable coefficients (size of `b1` tells the business the marginal effect of a feature). |


In [ ]:
# --- 2. Linear Regression on simulated single-feature data ---
X, y = make_regression(n_samples=120, n_features=1, noise=25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

lin = LinearRegression()
lin.fit(X_train, y_train)
y_pred = lin.predict(X_test)

r2_train = r2_score(y_train, lin.predict(X_train))
r2_test = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print(f"Slope (b1)    = {lin.coef_[0]:.3f}")
print(f"Intercept(b0) = {lin.intercept_:.3f}")
print(f"R^2 train     = {r2_train:.3f} | R^2 test = {r2_test:.3f}")
print(f"MAE = {mae:.2f} | RMSE = {rmse:.2f}  (predictions are, on average, RMSE dollars off)")

plt.figure(figsize=(9, 5))
plt.scatter(X_test, y_test, label="True", s=45, color="#2b6cb0")
plt.scatter(X_test, y_pred, label="Predicted", s=45, marker="x", color="#c53030")
plt.plot(X_test, y_pred, color="black", lw=2, label="Best-fit line")
plt.xlabel("Feature (e.g. size)")
plt.ylabel("Target (e.g. price)")
plt.title("Linear Regression — the line that comes 'closest' to all points")
plt.legend()
plt.show()

# Residuals = vertical gaps between true and predicted values
resid = y_test - y_pred
plt.figure(figsize=(9, 3.5))
plt.stem(X_test.ravel(), resid, basefmt=" ")
plt.axhline(0, color="black", lw=1)
plt.xlabel("Feature")
plt.ylabel("Residual (y - y_hat)")
plt.title(f"Residuals should be randomly scattered around 0  (MAE={mae:.2f})")
plt.show()


### Discussion — Regression (interview favourites)
- **"Why do we square the errors?"** → Squaring removes the sign (big over-predictions and
  under-predictions don't cancel out), punishes large errors more than small ones, and makes
  the maths smooth & differentiable. *Beginner tip: if errors are +100 then −100, the sum is
  0 and the average says "perfect" — that's wrong. Squaring turns both into +10,000.*
- **"R² = 0.9 — is that good?"** → It depends on the domain. R² can go negative for a model
  worse than predicting the mean. *Beginner tip: always compare R² of your model to a naive
  baseline (e.g. predicting the average target). A 0.9 R² on chaotic financial returns is
  suspicious — probably leakage or overfit.*
- **"What if the relationship is curved, not straight?"** → Linear regression would
  underfit (high **bias**). Two fixes: add polynomial features or switch to a non-linear
  model. This is the classic **bias–variance** trade-off explored in part 1.
- **"What's RMSE vs MAE?"** → Both are in the target units. MAE treats every error equally;
  RMSE (root mean squared error) punishes large errors more (squaring first). If RMSE ≫ MAE
  your data has big outlier errors.


## 3. Supervised · Classification — Logistic Regression

**Why it matters:** Classification (predicting a category) is the other half of supervised
learning and the most common interview topic. "Logistic regression" is *despite the name* a
**classifier**, and interviewers love that trap.

### 🧠 Beginner's guide
A **classifier** answers "which category?" — is this email **spam or not**, is this
transaction **fraud or legit**. Instead of drawing a line that predicts a number, Logistic
Regression:
1. draws a straight **decision boundary** through the features,
2. squashes every point's "score" through the **sigmoid** function `σ(z) = 1 / (1 + e^−z)` so
   the output is a **probability between 0 and 1**,
3. predicts class 1 when that probability **≥ 0.5** (the default threshold).

The threshold is the dial that trades off **precision** (few false alarms) vs **recall**
(catch more positives) — exactly what the confusion-matrix metrics from part 1 measure.

### ⭐ STAR — Logistic Regression
| Field | Content |
|---|---|
| **S**ituation | We must decide a **category** with uncertainty — e.g. "will this borrower default or not / is this email spam?" |
| **T**ask | Assign each point a class *and* a calibrated probability of belonging to the positive class. |
| **A**ction | Learn weights `w` so that probability `P(y=1) = σ(w·x + b)` is high for positives and low for negatives (log-loss objective). Map probability → class via a threshold (0.5 default). Evaluate with the confusion matrix, precision/recall/F1 and ROC-AUC. |
| **R**esult (**success criteria**) | **ROC-AUC clearly above 0.5** (0.8+ usually strong), well-calibrated probabilities, and a threshold chosen for the business priority (fraud → high recall; blocking legit customers → high precision). |


In [ ]:
# --- 3. Logistic Regression: decision boundary, sigmoid, confusion matrix ---
X, y = make_classification(n_samples=400, n_features=2, n_redundant=0, n_informative=2,
                           n_clusters_per_class=1, class_sep=1.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
y_proba = clf.predict_proba(X_te)[:, 1]

acc  = accuracy_score(y_te, y_pred)
prec = precision_score(y_te, y_pred)
rec  = recall_score(y_te, y_pred)
f1   = f1_score(y_te, y_pred)

# Decision boundary (probability surface)
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200),
                     np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200))
Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(9, 5))
plt.contourf(xx, yy, Z, levels=20, cmap="RdBu", alpha=0.6)
plt.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=2)
plt.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="coolwarm", edgecolor="k", s=30)
plt.title(f"Logistic Regression decision boundary  (Accuracy={acc:.3f}, F1={f1:.3f})")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

# Sigmoid: raw score on x-axis -> probability on y-axis
score = clf.decision_function(X_te)
plt.figure(figsize=(9, 3.5))
plt.scatter(score, y_proba, c=y_te, cmap="coolwarm", edgecolor="k", s=25)
plt.axhline(0.5, color="k", ls="--", lw=1, label="class threshold = 0.5")
plt.xlabel("Linear score (z)")
plt.ylabel("P(class = 1)")
plt.title("Sigmoid maps any score into a probability in [0, 1]")
plt.legend()
plt.show()

ConfusionMatrixDisplay.from_predictions(y_te, y_pred, cmap="Blues")
plt.title("Confusion Matrix (TN | FP / FN | TP)")
plt.show()
print(f"Precision={prec:.3f} | Recall={rec:.3f} | F1={f1:.3f}")


### ⭐ STAR — ROC & AUC (ranking quality)
| Field | Content |
|---|---|
| **S**ituation | Accuracy can be misleading; we want to know how well the model *ranks* positives above negatives at **all** thresholds, not just 0.5. |
| **T**ask | Summarise the precision/recall-type trade-off with one number that is threshold-free and class-imbalance-aware. |
| **A**ction | Plot the **ROC curve**: True Positive Rate (recall) vs False Positive Rate at every threshold. Compute the **Area Under the Curve (AUC)**. |
| **R**esult (**success criteria**) | **AUC = 0.5** is random guessing, **AUC = 1.0** is a perfect ranker, **AUC ≥ 0.8** is usually strong. A curve that bows up-left is always better than the diagonal. |


In [ ]:
# --- 3b. ROC curve & AUC ---
fpr, tpr, _ = roc_curve(y_te, y_proba)
auc = roc_auc_score(y_te, y_proba)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, lw=2, color="#2b6cb0", label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], ls="--", color="gray", label="Random guessing (AUC = 0.5)")
plt.fill_between(fpr, tpr, alpha=0.1, color="#2b6cb0")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR = Recall)")
plt.title("ROC curve — higher AUC = better at ranking positives first")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(f"ROC-AUC = {auc:.3f}  (0.5 = random, ~1.0 = perfect)")


### Discussion — Classification (interview favourites)
- **"Why is Logistic Regression a classifier if it has 'regression' in the name?"** → It
  *does* regress a continuous value — the log-odds / probability — but the output is then
  thresholded into a category. The name is historic (from statistics). *Beginner tip: the
  trained output is P(y=1), a number in [0,1]; the 'classification' happens when you apply a
  threshold.*
- **"How do you choose between precision and recall?"** → Depends on which error is more
  expensive. Spam: a few false alarms (precision) are okay, missing spam (recall) is also
  okay — balanced. Cancer screening: missing a case (low recall) is catastrophic → raise
  recall by lowering the threshold. *Beginner tip: precision answers 'when I say fraud, am I
  right?'; recall answers 'of all real fraud, how much did I catch?'.*
- **"Accuracy is 95% — impressive?"** → Not on imbalanced data. If 99 in 100 transactions
  are legit, a model that always says 'legit' scores 99% accuracy but catches no fraud. Use
  F1/ROC-AUC instead.
- **"What is the confusion matrix?"** → A 2×2 table of model mistakes: **TN** (correctly
  said no), **FP** (false alarm), **FN** (missed positive), **TP** (correctly caught). Every
  metric (precision, recall, F1, accuracy) is a ratio of these four numbers.


## 4. Supervised · Classification — k-Nearest Neighbours (k-NN)

**Why it matters:** k-NN is the easiest classification idea to understand and it exposes two
big interview themes: the **curse of dimensionality** and **overfitting/underfitting via the
hyperparameter k**.

### 🧠 Beginner's guide
k-NN is the "ask your neighbours" algorithm. To classify a new point:
1. measure its **distance** (e.g. Euclidean) to *every* training point,
2. keep the **k closest** neighbours,
3. let them **vote** — the majority class wins.

No training happens in the fitting step — it just *stores* the data (this is called a
**lazy / instance-based** learner). The magic knob is `k`:
- **small k (e.g. k=1)** → the model is very sensitive, memorises noise → **overfits**,
- **large k** → smooths decisions a lot, may miss local structure → **underfits**.

### ⭐ STAR — k-Nearest Neighbours
| Field | Content |
|---|---|
| **S**ituation | We have labelled examples and want to classify a new point using *similarity*, without assuming any functional form (e.g. linearity). |
| **T**ask | Predict the class of a new point by looking at the labelled examples nearest to it. |
| **A**ction | Pick the number of neighbours `k` (via cross-validation), choose a distance metric (Euclidean is the default), tally a majority vote among the k closest points. |
| **R**esult (**success criteria**) | Test accuracy is maximised at a moderate `k` (curve has a clear peak, not a plateaus/flat spike), and the model is *not* grossly overfit (train and test curves close together). If accuracy never gets high even for good `k`, the data may be too high-dimensional (curse of dimensionality). |


In [ ]:
# --- 4. k-NN: pick k via the accuracy curve, then visualise the boundary ---
X, y = make_moons(n_samples=300, noise=0.25, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

ks = range(1, 30, 2)
train_acc, test_acc = [], []
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    train_acc.append(accuracy_score(y_tr, knn.predict(X_tr)))
    test_acc.append(accuracy_score(y_te, knn.predict(X_te)))

plt.figure(figsize=(9, 4))
plt.plot(list(ks), train_acc, "o-", label="Train accuracy", color="#c53030")
plt.plot(list(ks), test_acc, "s-", label="Test accuracy", color="#2b6cb0")
plt.xlabel("k (number of neighbours)")
plt.ylabel("Accuracy")
plt.title("Choosing k: small k -> overfit (train high, test low); large k -> too smooth")
plt.legend()
plt.show()

best_k = ks[int(np.argmax(test_acc))]
knn = KNeighborsClassifier(n_neighbors=best_k).fit(X_tr, y_tr)
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200),
                     np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200))
Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, cmap="coolwarm", alpha=0.5)
plt.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="coolwarm", edgecolor="k", s=25)
plt.title(f"k-NN decision boundary (best k = {best_k}, test acc = {max(test_acc):.3f})")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()
print(f"Best k = {best_k}  |  max test accuracy = {max(test_acc):.3f}")


### Discussion — k-NN (interview favourites)
- **"What is the curse of dimensionality?"** → In high-dimensional spaces, points become
  roughly *all equidistant* — every point is about as far from every other, so "nearest
  neighbours" stop being meaningful. k-NN degrades badly. *Beginner tip: think of a 1000-D
  cube — almost all of its volume sits at the corners. Distance loses its power to separate
  points.*
- **"Is k-NN fast to train?"** → Training is nearly instant (it just stores data), but
  **prediction is slow** because it must scan all points and compute distances every time.
  That's the opposite of a neural net (slow to train, fast to predict).
- **"How do you choose k and the distance metric?"** → Try a range of `k` with
  cross-validation and pick the one with best validation score. Try Euclidean vs Manhattan
  distance; scale the features first so large-magnitude features don't dominate the distance.
- **"Why scale features before k-NN?"** → Distance mixes all features. If "salary" ranges up
  to 1,000,000 and "age" to 90, salary dwarfs age in the distance → age has no influence.
  Scaling (z-score / min-max) puts every feature on an equal footing.


## 5. Supervised · Classification — Decision Trees & Random Forest

**Why it matters:** Decision trees are the most *interpretable* ML models (you can literally
read the rules). Random Forest fixes trees' biggest weakness (overfitting) and is a
go-to model for tabular data — a favourite "which model should I try first?" answer.

### 🧠 Beginner's guide
A **decision tree** is a flowchart of `if … then` questions: "if income > 50k and age < 30
then class A". At each node it picks the feature and split that best separates the classes —
measured by **Gini impurity** or **entropy** (how "mixed" a node still is). It recurses down
until pure leaves or a stopping rule (`max_depth`) is reached.

A single tree **overfits** easily (it memorises noise). A **Random Forest** uses a trick
called **bagging**: train *many* trees, each on a random sample of the data (bootstrap) and a
random subset of features, then **average their votes**. Averaging reduces variance — the
whole is more robust than any single tree.

### ⭐ STAR — Decision Tree & Random Forest
| Field | Content |
|---|---|
| **S**ituation | The data is non-linear with feature interactions, and we want both good performance *and* explainability (regulated settings favour readable rules). |
| **T**ask | Learn a set of if-then rules that separate the classes and generalise to unseen data. |
| **A**ction | **Tree:** recursively split on the feature/split minimising impurity (Gini/entropy), pruning with `max_depth`. **Forest:** bag many random trees (bootstrap samples + random feature subsets) and average their votes to cut variance and overfitting. |
| **R**esult (**success criteria**) | Random Forest attains higher, more stable test accuracy than a single tree (and beats logistic regression on non-linear data); the tree stays *readable* (shallow depth); feature importances highlight which inputs drive the decision. |


In [ ]:
# --- 5. Decision Tree vs Random Forest on non-linear data ---
X, y = make_classification(n_samples=800, n_features=8, n_informative=5, n_redundant=2,
                           n_repeated=0, class_sep=1.0, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

tree   = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
forest = RandomForestClassifier(n_estimators=120, random_state=42).fit(X_tr, y_tr)

tree_acc = accuracy_score(y_te, tree.predict(X_te))
forest_acc = accuracy_score(y_te, forest.predict(X_te))
print(f"Single tree (depth=3)  test accuracy: {tree_acc:.3f}")
print(f"Random forest (120 trees) test accuracy: {forest_acc:.3f}")

# Feature importances: which features matter most (built into RandomForest)
imp = forest.feature_importances_
plt.figure(figsize=(9, 4))
plt.bar(range(1, 9), imp, color="#2b6cb0")
plt.xticks(range(1, 9), [f"F{i}" for i in range(1, 9)])
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.title("Random Forest feature importances — read the tree's 'opinions'")
plt.show()

# A shallow tree is fully readable
plt.figure(figsize=(15, 6))
plot_tree(tree, filled=True, feature_names=[f"F{i}" for i in range(1, 9)], fontsize=9)
plt.title("A single decision tree (depth = 3) — readable if-then rules")
plt.show()


### Discussion — Trees & Ensembles (interview favourites)
- **"Why does Random Forest beat a single decision tree?"** → A single deep tree has **high
  variance** (change the data slightly → very different tree). Averaging many trees that each
  see a different bootstrap sample + feature subset reduces variance hugely, while keeping
  low bias. *Beginner tip: one expert can be confidently wrong; 100 experts with different
  blind-spots voting together are far more reliable.*
- **"What is Gini impurity / entropy?"** → They both measure how "mixed" a node is. Gini =
  1 − Σ p². A node that is half-and-half (0.5, 0.5) has max impurity; an all-one-class node
  has impurity 0. At each split we choose the split that reduces impurity the most.
  *Beginner tip: entropy is the 'surprise' measure; Gini is simpler to compute. Both guide
  the same greedy search.*
- **"Do trees need feature scaling?"** → No. Trees split on *thresholds* feature-by-feature,
  so scale doesn't affect them. That's a real advantage over k-NN and linear/neural models.
- **"What hyperparameters prevent overfitting?"** → `max_depth`, `min_samples_split`, and
  `min_samples_leaf` (constrain tree size), plus the number of trees `n_estimators` and
  feature sampling in a forest. Tune these with cross-validation.
- **"Tree-based vs neural networks?"** → Trees/Random Forest/XGBoost dominate **tabular**
  data; neural networks win on **images, audio and text**. For a finance-tabular problem,
  start with a forest or gradient boosting.


## 6. Validation — train/test split & cross-validation

**Why it matters:** The #1 skill interviewers probe is whether you can *honestly* estimate
how well your model will do in the real world. Training accuracy is almost always too
optimistic. Validation is how you keep yourself — and them — honest.

### 🧠 Beginner's guide
- **Train / validation / test split:** a model should never be tuned or evaluated on the same
  data it learned from. Standard practice: fit on **train**, choose hyperparameters on
  **validation**, report final numbers on a **test set it never saw**.
- **Cross-validation (k-fold):** split *training* data into `k` folds; train on k−1 folds,
  evaluate on the remaining 1, repeat k times, average. This uses all data for both training
  and validation → a **more stable, less lucky** performance estimate than a single split.
- **The golden rule:** "no peeking" — leakage (using test info in training, e.g. scaling the
  test set before splitting) makes numbers look great but fail in production.

### ⭐ STAR — Validation
| Field | Content |
|---|---|
| **S**ituation | A model scores 99% on training data but we don't know if it generalises — single splits can be fluky and small datasets waste data. |
| **T**ask | Get an unbiased, stable estimate of real-world performance and detect overfitting. |
| **A**ction | Hold out a **test set** (never touched during tuning). Use **k-fold cross-validation** (e.g. `cv=5`) on train data to tune hyperparameters and average fold scores. Compare train vs validation accuracy to spot the gap. |
| **R**esult (**success criteria**) | CV mean ± std is tight (stable), train and validation scores are close (little overfitting), and the final test score is close to the CV estimate. A big train≫val gap = overfitting → simplify/regularise. |


In [ ]:
# --- 6. One split vs 5-fold cross-validation ---
X, y = make_classification(n_samples=500, n_features=6, n_informative=4, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
single_test = accuracy_score(y_te, model.predict(X_te))

scores = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42), X_tr, y_tr, cv=5
)
print(f"Single 25%-holdout test accuracy : {single_test:.3f}")
print(f"5-fold CV mean +/- std          : {scores.mean():.3f} +/- {scores.std():.3f}")

plt.figure(figsize=(9, 4))
sns.boxplot(x=scores, color="#2b6cb0")
sns.stripplot(x=scores, color="k", jitter=True, size=7)
plt.title("Accuracy across 5 folds — how stable / lucky is the estimate?")
plt.xlabel("Fold accuracy")
plt.show()
print("The CV spread tells you whether that single holdout number was luck.")


### Discussion — Validation (interview favourites)
- **"Why not just use one train/test split?"** → One split is a single draw of luck: the test
  set's composition can flatter or punish a model. k-fold CV averages over k different splits
  → far more reliable, and it reuses all data. *Beginner tip: with small data, a single 80/20
  split throws away 20% of training signal; k-fold uses everything.*
- **"What is data leakage?"** → Using validation/test information during training — e.g.,
  `fit`ing the scaler on the *whole* dataset before splitting, or letting a time-based test
  set leak future values into training. *Beginner tip: apply every transformation only AFTER
  splitting, and fit scalers/encoders on train and `.transform` the test.*
- **"When is test accuracy not enough?"** → When classes are imbalanced (use ROC-AUC/F1), when
  you care about a specific error (use precision/recall), or when the model must be
  explainable (report feature importance too). Also always compare against a **baseline**:
  "beats random / beats the current rule-based system."
- **"What's a reasonable CV setup?"** → 5- or 10-fold CV for general data; for **time series**,
  use time-based splits (train on the past, validate on the future) because random shuffling
  would leak the future — a classic finance interview gotcha.


## 7. Unsupervised · Clustering — k-Means

**Why it matters:** Clustering is the first unsupervised task and a safe interview answer
when the data has no labels ("we'll segment customers, not predict churn"). k-Means is the
classic first clustering algorithm.

### 🧠 Beginner's guide
Given points and a chosen `k`, k-Means:
1. places `k` **centroids** (cluster centres) randomly,
2. **assigns** every point to its nearest centroid,
3. **updates** each centroid to the mean of its assigned points,
4. repeats 2–3 until nothing changes.

Goal: minimise **inertia** = the total squared distance from each point to its centroid
(within-cluster sum of squares). Two real questions:
- **How to pick k?** → the **elbow method** (plot inertia vs k; take the "knee") and the
  **silhouette score** (higher = points are far from the other clusters / near their own).
- **Know its limits** → k-Means assumes round, similarly-sized clusters; it fails on the
  thin/curvy "moons" shapes we'll see with DBSCAN next.

### ⭐ STAR — k-Means clustering
| Field | Content |
|---|---|
| **S**ituation | A marketing team has customer profiles but **no labels** — they want natural segments to target, but nobody has said how many segments exist. |
| **T**ask | Partition customers into `k` groups such that members of a group are similar and groups are distinct. |
| **A**ction | Run k-Means for several `k` values, choose `k` via the **elbow** (inertia) and **silhouette** criteria, then interpret each cluster (e.g. "high-value, low-activity"). |
| **R**esult (**success criteria**) | Inertia drops sharply to an elbow then flattens; **silhouette close to 1** (ideally ≥ 0.5 for reasonably separated clusters) across chosen `k`; clusters are stable and business-interpretable. Low silhouette (< 0.25) warns data may not cluster well with k-Means. |


In [ ]:
# --- 7. k-Means: pick k via elbow + silhouette, then view the clusters ---
X, _ = make_blobs(n_samples=500, centers=4, cluster_std=1.2, random_state=42)

inertia, silhouettes = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertia.append(km.inertia_)
    silhouettes.append(silhouette_score(X, km.labels_))

plt.figure(figsize=(9, 4))
plt.plot(list(K_range), inertia, "o-", color="#2b6cb0")
plt.xlabel("k (number of clusters)")
plt.ylabel("Inertia (within-cluster sum of squares)")
plt.title("Elbow method — look for the 'knee' (here it is near k = 4)")
plt.show()

plt.figure(figsize=(9, 4))
plt.bar(list(K_range), silhouettes, color="#38a169")
plt.axhline(0.5, color="gray", ls="--", lw=1, label="0.5 = fairly separated")
plt.xlabel("k")
plt.ylabel("Silhouette score")
plt.title("Silhouette score — higher is better (-1..+1)")
plt.legend()
plt.show()

km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X)
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap="viridis", edgecolor="k", s=30)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            marker="X", s=220, c="red", label="centroids")
plt.title(f"k-Means clusters (silhouette = {silhouette_score(X, km.labels_):.3f})")
plt.legend()
plt.show()


### Discussion — k-Means (interview favourites)
- **"How do you choose k?"** → Elbow on the inertia curve, maximise silhouette score, and
  — very importantly — **business judgment**: does the number of segments make sense and is it
  actionable? *Beginner tip: k is your own choice; two people often pick different k and both
  can be "valid" — the real test is whether the segments are useful and stable.*
- **"What are k-Means' assumptions / limitations?"** → It assumes clusters are roughly
  **spherical, convex and similar in size**, and it's sensitive to the initial centroids
  (use `n_init` to run several restarts). It also can't handle arbitrary shapes or outliers
  well. *Beginner tip: the moment you see crescent-shaped groups, k-Means silently splits each
  crescent in half — that's exactly what DBSCAN fixes.*
- **"What is the silhouette score?"** → For each point: (distance to its *own* cluster's
  boundary − distance to the *nearest other* cluster), normalised to [−1, +1]. +1 = perfectly
  separated, 0 = on the boundary, −1 = wrong cluster. Average it over all points.
- **"K-Means never converges to a good solution. What next?"** → Try k-Means++ init (default in
  sklearn), more restarts, standardise features first, or switch to GMM/DBSCAN/hierarchical.


## 8. Unsupervised · Clustering — DBSCAN (density-based)

**Why it matters:** DBSCAN is the antidote to k-Means' blind spots. Interviewers love to ask
"what if clusters aren't round?" — DBSCAN is the answer, and it also handles outliers and
needs **no k**.

### 🧠 Beginner's guide
Instead of choosing k centroids, DBSCAN looks at **density**. Around each point it checks a
sphere of radius **eps** containing at least **min_samples** points:
- a **core point** has ≥ `min_samples` neighbours inside `eps`,
- points reachable from a core point, through other core points, join the same cluster,
- points that can't be reached from any core point are **noise** (`label = −1`).

So it can find clusters of **arbitrary shape**, it automatically decides the number of
clusters, and it flags outliers — none of which k-Means can do.

### ⭐ STAR — DBSCAN
| Field | Content |
|---|---|
| **S**ituation | Customer/transaction data forms **crescent or long, curvy shapes** (e.g. two demand bands) plus stray noise points; k-Means would split each band in half and be fooled by outliers. |
| **T**ask | Group points by *density* into arbitrarily-shaped clusters while identifying genuine outliers — without knowing k in advance. |
| **A**ction | Choose `eps` (neighbourhood radius) and `min_samples`; mark dense regions as clusters and sparse points as noise. Tune `eps` with a k-distance plot. |
| **R**esult (**success criteria**) | The two crescent bands are recovered as exactly 2 clusters (k-Means fails, DBSCAN succeeds), noise is flagged separately, and the recovered structure is stable to small changes in `eps`/`min_samples`. |


In [ ]:
# --- 8. DBSCAN vs k-Means on crescent-shaped clusters (the same 'moons') ---
X, _ = make_moons(n_samples=400, noise=0.07, random_state=42)

km = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X)   # k-Means forced to 2 groups
db = DBSCAN(eps=0.2, min_samples=8).fit(X)                     # DBSCAN finds density + noise

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X[:, 0], X[:, 1], c=km.labels_, cmap="coolwarm", edgecolor="k", s=30)
axes[0].set_title("k-Means (assumes round clusters):\nsplits each crescent in half — wrong")
axes[1].scatter(X[:, 0], X[:, 1], c=db.labels_, cmap="coolwarm", edgecolor="k", s=30)
axes[1].set_title("DBSCAN (density-based):\nfollows the two crescents; gray + = noise")
for ax in axes:
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
plt.tight_layout()
plt.show()

n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print("Clusters found by DBSCAN:", n_clusters)
print("Noise points (label = -1):", int((db.labels_ == -1).sum()))


### Discussion — DBSCAN (interview favourites)
- **"When would you pick DBSCAN over k-Means?"** → When clusters are **non-spherical /
  arbitrarily shaped**, when the data has **outliers** you want to detect, or when you don't
  want to commit to a `k`. If clusters are round and equal-sized, k-Means is simpler and
  faster. *Beginner tip: 'density' approaches shine wherever 'distance to a single centre' is
  the wrong notion — e.g. geographic hotspots, anomaly detection, elongated traffic bands.*
- **"What are eps and min_samples?"** → `eps` = radius of the neighbourhood; `min_samples` =
  how many neighbours are needed for a point to be "core". Too small `eps` → everything is
  noise; too large → everything is one cluster. *Beginner tip: that trade-off is DBSCAN's
  version of k-NN's k, and it must be tuned on a validation criterion.*
- **"Does DBSCAN always find a good number of clusters?"** → No — it's sensitive to `eps` and
  struggles when clusters have **very different densities** (one dense band + one sparse band).
  For varying densities, HDBSCAN (hierarchical variant) is a common upgrade.
- **"How do you pick eps?"** → Sort each point's distance to its `min_samples`-th nearest
  neighbour and find the "knee" in that k-distance plot — points beyond the knee are the
  sparse/noise ones. Same elbow-style reasoning as k-Means.


## 9. Unsupervised · Clustering — Hierarchical (agglomerative)

**Why it matters:** Sometimes a flat "k clusters" answer isn't enough — you want a *family
tree* of groupings (e.g. species taxonomy, org charts, "which countries are closest"). That
is exactly what hierarchical clustering gives you, and it produces the beautiful
**dendrogram** every interviewer recognises.

### 🧠 Beginner's guide
**Agglomerative** (bottom-up) clustering:
1. start with **every point as its own cluster**,
2. repeatedly **merge the two closest clusters** (using a *linkage* rule),
3. continue until one cluster remains.

The result is a tree (**dendrogram**). To get `k` clusters, you simply **cut** the tree at a
chosen height — the branches below the cut are your clusters. This means you can get any
number of clusters from a *single* run, which is a big practical advantage.

Common **linkage** rules for "closest":
- **ward** — minimise the increase in within-cluster variance (usually the default, balanced),
- **single** — closest pair of points (finds long chains; sensitive to outliers),
- **complete** — farthest pair of points (strict, rounder clusters),
- **average** — average pair distance (a middle ground).

### ⭐ STAR — Hierarchical clustering
| Field | Content |
|---|---|
| **S**ituation | We need an *explorable* grouping where the number of clusters isn't fixed in advance and we want to see how groups relate at every granularity (e.g. segment regions: districts → cities → countries). |
| **T**ask | Build a nested hierarchy of groups and pick a cut that yields interpretable clusters. |
| **A**ction | Agglomeratively merge the closest clusters into a **dendrogram**, then choose the cut height (and linkage) that gives clean, separable groups. |
| **R**esult (**success criteria**) | The dendrogram shows clear, distant merge levels (tall gaps before big merges = natural structure); cutting it yields compact, separated clusters; the grouping is robust to the chosen linkage. Very tall "gluey" merges suggest there's little real clustering. |


In [ ]:
# --- 9. Hierarchical clustering: build a dendrogram, then cut it ---
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

X, _ = make_blobs(n_samples=60, centers=3, cluster_std=1.0, random_state=42)
Z = linkage(X, method="ward")   # each row merges the two closest clusters

plt.figure(figsize=(12, 6))
dendrogram(Z, truncate_mode="level", p=8, color_threshold=8.0)
plt.title("Dendrogram — read bottom-up: merges; cut height -> number of clusters")
plt.xlabel("Sample index (grouped by cluster)")
plt.ylabel("Distance / dissimilarity")
plt.show()

labels = fcluster(Z, t=3, criterion="maxclust")   # cut the tree to get exactly 3 clusters
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="viridis", edgecolor="k", s=50)
plt.title("3 clusters obtained by cutting the dendrogram at height = 3")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()


### Discussion — Hierarchical clustering (interview favourites)
- **"What is the difference between divisive and agglomerative?"** → **Agglomerative**
  (bottom-up) merges the two closest clusters each step. **Divisive** (top-down) starts with
  one big cluster and splits it. Agglomerative is far more common.
- **"How do you pick the final k from a dendrogram?"** → Look for the **largest vertical
  distance** you can cut without splitting a natural group — the "biggest gap" between merge
  levels. Then verify the chosen clusters with silhouette / domain sense. *Beginner tip: a
  clean dendrogram has several long, tall merges; a ladder of short mergers means weak
  structure.*
- **"Single vs ward linkage — what's the trade-off?"** → **Single** can create long chains
  and joins clusters through a single point (bad with outliers); **ward** tends to make
  compact, near-spherical clusters and is a sensible default for continuous features.
- **"Why use hierarchical instead of k-Means?"** → You get a full hierarchy (k is decided
  after the fact, not before), and a visual to present to business. Cost: it's **O(n²)** time
  and memory, so it struggles with very large datasets — k-Means scales much better.


## 10. Unsupervised · Dimensionality Reduction — PCA

**Why it matters:** Real data often has hundreds of correlated features. **Principal
Component Analysis (PCA)** is the go-to unsupervised technique to compress them — and it
doubles as the standard tool for **visualising high-dimensional data**.

### 🧠 Beginner's guide
PCA finds **new axes (components)** pointing in the directions of *maximum variance* in the
data, subject to each new axis being perpendicular to the previous ones. Then it **projects**
the data onto the first few components:
- **PC1** captures the most variance, **PC2** the next-most, and so on,
- because components are perpendicular (uncorrelated), PCA also **decorrelates** features,
- you *drop* the low-variance components → you lose little information while shrinking the
  feature count dramatically.

The **explained variance ratio** tells you how much of the total information each component
keeps. A common rule: keep as many components as needed to explain ~95% of variance (or just
2–3 to plot).

### ⭐ STAR — PCA
| Field | Content |
|---|---|
| **S**ituation | A model has 6–100 highly correlated features; training is slow, k-NN distances are unstable, and we literally cannot *see* the data in 6 dimensions. |
| **T**ask | Reduce the number of features while keeping most of the information, and make the data visualisable. |
| **A**ction | **Standardise** features (PCA is scale-sensitive), compute the covariance/eigen-decomposition to get principal components, project data onto the top components, and verify with the explained-variance ratio. |
| **R**esult (**success criteria**) | A handful of components (2–3) capture ~95% of variance (or ≥ 80–90% for a large drop in dimensions); reconstruction error is low; the 2-D projection visually separates the natural groups; downstream model performance stays roughly equal to the full-feature version. |


In [ ]:
# --- 10. PCA: how many components, then project to 2D ---
X, y = make_classification(n_samples=300, n_features=6, n_informative=2, n_redundant=4,
                           random_state=42)
Xs = StandardScaler().fit_transform(X)   # PCA is very sensitive to feature scale

pca = PCA().fit(Xs)
evr = pca.explained_variance_ratio_

plt.figure(figsize=(9, 4))
plt.bar(range(1, 7), evr * 100, color="#2b6cb0", label="per component")
plt.plot(range(1, 7), np.cumsum(evr) * 100, "ro-", label="cumulative %")
plt.axhline(95, color="gray", ls="--", lw=1, label="95% threshold")
plt.xlabel("Principal component")
plt.ylabel("Variance explained (%)")
plt.title("Scree plot — top components carry most information")
plt.legend()
plt.show()

X2 = PCA(n_components=2).fit_transform(Xs)
plt.figure(figsize=(8, 6))
plt.scatter(X2[:, 0], X2[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=35)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Original 6-D data projected onto 2 principal components\n(now we can actually SEE it)")
plt.show()

print(f"Top 2 PCs explain {evr[:2].sum()*100:.1f}% of the variance "
      f"(need only {np.argmax(np.cumsum(evr) >= 0.95) + 1} PCs to reach 95%).")


### Discussion — PCA (interview favourites)
- **"Do you always need PCA?"** → No. Use it for **visualisation**, for **removing
  multicollinearity**, or when the feature count is large vs the number of samples. Tree-based
  models often don't need it (they don't care about correlation the same way) and you may lose
  interpretability — components are *combinations* of features, not original features.
- **"Why standardise before PCA?"** → PCA maximises variance; if one feature has a huge scale
  (income 0–10⁶) and another is small (age 0–100), the big one dominates the components purely
  due to its units. Standard scaling (z-score) puts them on equal footing.
- **"What do the components mean?"** → Each component is a **weighted combination of the
  original features** (its "loadings"). A large loading means that feature contributes strongly
  to that component. You can name a component ("overall credit-worthiness") if its loadings
  align meaningfully.
- **"PCA is unsupervised — can it still help supervised tasks?"** → Yes. It's an **unsupervised
  dimensionality-reduction preprocessing step**; you still pass the compressed features to a
  supervised classifier. It often speeds training and can reduce overfitting, at the cost of
  some explainability.


## 11. Cheat-sheet & interview recap

Use this table to answer the classic opener **"if I gave you this problem, how would you
start?"** — pick family → algorithm → metric.

| Problem (⭐ Situation) | Family | Suggested approach ⭐ Action | Success criteria (⭐ Result) |
|---|---|---|---|
| Predict a **number** (price, revenue, demand) | Supervised · Regression | Linear Regression baseline → Ridge/Lasso → Random Forest / XGBoost | R² far above 0, low MAE/RMSE, residual plot flat |
| Predict a **category** (spam, fraud, default) | Supervised · Classification | Logistic Regression → Random Forest → XGBoost | ROC-AUC > 0.8, F1/precision/recall aligned to business |
| Classify by **similarity**, few assumptions | Supervised · Classification | k-NN with tuned k → (scale features) | Peak in accuracy-vs-k curve, train≈test |
| Need **interpretable rules** / tree logic | Supervised · Classification | Shallow Decision Tree, Random Forest for accuracy | Readable tree, feature importances, high test accuracy |
| **No labels** — discover groups | Unsupervised · Clustering | k-Means (round) → Hierarchical (hierarchy) → DBSCAN (odd shapes/noise) | High silhouette, elbow at chosen k, stable clusters |
| **Too many features** / can't visualise | Unsupervised · Dimensionality reduction | PCA (then keep ~95% variance) | ~2–3 PCs explain ≈95% variance, low reconstruction error |
| Any supervised model | — | Train/test split + k-fold CV (+ no leakage) | CV score ≈ test score, tight std, train≈val |

### The STAR "elevator pitch" you can reuse
> **S**ituation (real problem) → **T**ask (ML formulation) → **A**ction (data prep + model +
> validation plan) → **R**esult (the numeric success criteria and how you'd know it's met).

If you can fill that template for every algorithm in sections 1–10, you have answered the
majority of entry-level ML interview questions. Good luck — and always **question the data
before you trust a number**. 🚀
